# Predictive Analysis of Stock Market Trends

## Project Overview
This study aims to develop a robust machine learning framework to predict short-term stock price movements. Instead of forecasting exact prices—which are highly susceptible to market noise—this project focuses on **binary classification**: predicting whether a stock's closing price will move **UP** or **DOWN** on the following trading day.

### Why Classification?
1. **Reliability**: Captures the underlying trend rather than focusing on precise numeric values.
2. **Risk Management**: Providing a directional signal is often more actionable for decision-making.
3. **Robustness**: Machine learning models often generalize better to directional patterns in volatile financial data.

## 1. Environment Configuration
We utilize a standard data science stack including `pandas` for data manipulation, `scikit-learn` for machine learning, and `yfinance` for real-time market data access.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

## 2. Data Acquisition
We retrieve historical market data to establish a foundation for our analysis. The dataset includes Open, High, Low, Close, and Volume (OHLCV) metrics.

In [ ]:
# Define the ticker symbol and time range
ticker = 'AAPL'
start_date = '2015-01-01'
end_date = '2024-01-01'

# Fetch the data
print(f"Fetching data for {ticker}...")
df = yf.download(ticker, start=start_date, end=end_date)

# Flatten MultiIndex columns if present
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

# Display the structure
display(df.head())
print("\nDataset Info:")
display(df.info())

# Save raw data
df.to_csv(f'../data/{ticker}_historical_data.csv')

## 3. Exploratory Data Analysis (EDA)
Visual analysis is critical to understanding historical price trends, identifying periods of high volatility, and detecting anomalous trading behavior.

In [ ]:
# Set a clean, professional visualization style
plt.style.use('bmh')
sns.set_palette("viridis")

# 1. Plot Closing Price History
plt.figure(figsize=(14, 7))
plt.plot(df.index, df['Close'], label='Close Price', color='#2ca02c')
plt.title(f'{ticker} Historical Closing Price (2015 - 2024)', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend()
plt.show()

# 2. Interactive Candlestick Chart (Detailed view)
fig = go.Figure(data=[go.Candlestick(x=df.index,
                open=df['Open'],
                high=df['High'],
                low=df['Low'],
                close=df['Close'])])
fig.update_layout(title=f'{ticker} Interactive Market Structure',
                  yaxis_title='Price (USD)',
                  xaxis_rangeslider_visible=False,
                  template='plotly_white')
fig.show()

In [ ]:
# 3. Volume over Time Analysis
plt.figure(figsize=(14, 5))
plt.fill_between(df.index, df['Volume'], color='#1f77b4', alpha=0.3)
plt.plot(df.index, df['Volume'], color='#1f77b4', linewidth=0.5)
plt.title(f'{ticker} Historical Trading Volume', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Volume (Units)')
plt.show()

### EDA Observations:
*   **Trend Analysis**: The asset shows a consistent long-term upward trajectory with distinct growth phases.
*   **Volatility**: Significant price swings are often correlated with spikes in trading volume, indicating high market interest.
*   **Market Noise**: Daily fluctuations highlight the difficulty of predicting exact prices, reinforcing the decision to use directional classification.

## 4. Advanced Feature Engineering
Raw market prices are often insufficient for machine learning. We engineer "Technical Indicators" to capture trends, momentum, and volatility signals that models can interpret more effectively.

In [ ]:
def calculate_rsi(data, window=14):
    delta = data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

# Trend Indicators
df['SMA_20'] = df['Close'].rolling(window=20).mean()
df['SMA_50'] = df['Close'].rolling(window=50).mean()
df['EMA_20'] = df['Close'].ewm(span=20, adjust=False).mean()

# Momentum Indicators
df['RSI_14'] = calculate_rsi(df)

# Risk & Return Indicators
df['Daily_Return'] = df['Close'].pct_change()
df['Volatility_20'] = df['Daily_Return'].rolling(window=20).std()

# TARGET GENERATION (Classification: 1 if Next Close > Current Close, else 0)
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Data Integrity: Drop NaN values from rolling calculations
df.dropna(inplace=True)

print("Feature Engineering Complete.")
display(df[['Close', 'SMA_20', 'RSI_14', 'Target']].tail())

### Why These Features?
*   **Moving Averages (SMA/EMA)**: Smooth out noise and identify the prevailing market trend direction.
*   **RSI**: Identifies "overbought" or "oversold" conditions which may precede price reversals.
*   **Volatility**: Provides context on the current risk environment, which heavily influences model confidence.

## 5. Predictive Modeling and Performance Evaluation
We utilize a **Random Forest Classifier**—an ensemble learning method that builds multiple decision trees and merges them together to get a more accurate and stable prediction.

In [ ]:
# 1. Feature Selection
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_20', 'RSI_14', 'Daily_Return', 'Volatility_20']
X = df[features]
y = df['Target']

# 2. Chronological Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Model Initialization & Training
print("Training Random Forest Classifier...")
model = RandomForestClassifier(n_estimators=200, min_samples_split=50, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 4. Inference
preds = model.predict(X_test)

# 5. Statistical Evaluation
accuracy = accuracy_score(y_test, preds)
print(f"\nOverall Model Accuracy: {accuracy:.2%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, preds))

## 6. Analytical Insights and Visualizations
To validate our results academically, we analyze the **Confusion Matrix** (to understand error types) and **Feature Importance** (to explain *why* the model makes specific decisions).

In [ ]:
# 1. Confusion Matrix Visualization
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Price DOWN', 'Price UP'])
disp.plot(cmap='Blues', ax=ax)
plt.title('Prediction Accuracy Breakdown (Confusion Matrix)')
plt.grid(False)
plt.show()

# 2. Feature Importance Analysis
importance = pd.Series(model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importance.plot(kind='barh', color='teal')
plt.title('Feature Importance: What Influences Predictions?', fontsize=15)
plt.xlabel('Relative Importance Score')
plt.ylabel('Technical Indicator')
plt.show()

## Final Project Conclusions
1. **Model Performance**: The classification approach provides a more realistic assessment of market predictability compared to regression.
2. **Critical Signals**: The **Feature Importance** graph reveals the indicators that played the most significant role in our predictions.
3. **Future Directions**: To improve accuracy, future iterations could incorporate sentiment analysis from news sources or explore Deep Learning architectures like LSTMs.